# AB Testing Ecommerce — Pipeline Walkthrough

> Two-proportion z-test on a landing-page experiment.

This notebook walks through the production pipeline using the modules in `src/`. The model is loaded from the serialized artifact produced by `python -m src.pipeline`.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_columns", 50)

## 1. Data

Load the versioned sample and inspect it.

In [2]:
df = pd.read_csv(config.SAMPLE_PATH)
print(df.shape)
df.head()

(30000, 5)


,user_id,timestamp,group,landing_page,converted
0,631010,2017-01-18 04:44:18.590293,control,old_page,0
1,736681,2017-01-14 05:22:43.771772,control,old_page,0
2,916949,2017-01-23 08:52:53.050325,control,old_page,0
3,634217,2017-01-07 23:27:09.109877,control,old_page,0
4,932468,2017-01-20 14:26:10.133979,control,old_page,0


## 2. Preprocessing

The same transform used in training and serving.

In [3]:
from src.preprocessing import Preprocessor
clean = Preprocessor().run(df)
print("clean rows:", len(clean))
clean.groupby(config.GROUP_COL)[config.CONVERT_COL].mean().round(4)

clean rows: 29600


group
control      0.1288
treatment    0.1219
Name: converted, dtype: float64

## 3. Model and evaluation

Metrics from the serialized model card.

In [4]:
card = json.loads(Path(config.MODEL_CARD_PATH).read_text())
print(json.dumps(card, indent=2)[:1800])

{
  "schema_version": "1.0",
  "created_at": "2026-06-14T14:48:06+00:00",
  "dataset": "zhangluyuan/ab-testing",
  "data_sha256": "d56e2accec25e99ac21cb3d76c5df516dd19cc7a77c14c9014f94e1ea1301beb",
  "problem": "A/B test: does the new landing page change conversion?",
  "result": {
    "groups": {
      "control": {
        "n": 145274,
        "conversions": 17489,
        "rate": 0.12039
      },
      "treatment": {
        "n": 145309,
        "conversions": 17264,
        "rate": 0.11881
      }
    },
    "test": {
      "name": "Two-proportion z-test",
      "z_statistic": -1.3102408579271012,
      "p_value": 0.19011436776805013,
      "abs_difference": -0.0015774213617705535,
      "relative_difference": -0.013102996792832946,
      "ci95_difference": [
        -0.003937093508865433,
        0.0007822507853243259
      ],
      "alpha": 0.05,
      "observed_power": 0.2585,
      "significant": false,
      "decision": "Fail to reject H0: no significant difference; keep the cu

## 4. Prediction

The serving contract on a representative input.

In [5]:
from src.predict import Predictor
pred = Predictor()
print("test:", pred.test().get("decision"))
print("required n/group to detect +1pp at 12% baseline:", pred.sample_size(0.12, 0.01))

test: Fail to reject H0: no significant difference; keep the current page.
required n/group to detect +1pp at 12% baseline: 17169


## Reproduce

Run the full pipeline end to end:

```
python -m src.pipeline
```